In [1]:
from tensorflow.keras import models, layers, utils
from sqlite3 import connect
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import os
import tf2onnx
import onnx
import tensorflow as tf

db = connect('../StocksPlatform/app.db')

In [2]:
prices = pd.read_sql('SELECT AssetId, Timestamp, Price, Volume FROM AssetDailyHistory', db)
assets = pd.read_sql('SELECT Id, Symbol, Country FROM Assets', db)
prices = prices.merge(assets, left_on='AssetId', right_on='Id', how='left').drop(columns=['Id'])
prices

,AssetId,Timestamp,Price,Volume,Symbol,Country
0,2C586544-7C13-625C-AB97-C0DE16AECF5D,2021-03-15 00:00:00,235.979995727539,88006800,TSLA,United States
1,2C586544-7C13-625C-AB97-C0DE16AECF5D,2021-03-16 00:00:00,225.626663208008,96587100,TSLA,United States
2,2C586544-7C13-625C-AB97-C0DE16AECF5D,2021-03-17 00:00:00,233.936660766602,121117500,TSLA,United States
3,2C586544-7C13-625C-AB97-C0DE16AECF5D,2021-03-18 00:00:00,217.720001220703,99674400,TSLA,United States
4,2C586544-7C13-625C-AB97-C0DE16AECF5D,2021-03-19 00:00:00,218.289993286133,128682000,TSLA,United States
...,...,...,...,...,...,...
1118594,84879EDE-767B-DC5E-A377-7C3F92603B1C,2026-03-16 00:00:00,50.439998626709,4824300,FIS,United States
1118595,84879EDE-767B-DC5E-A377-7C3F92603B1C,2026-03-17 00:00:00,50.2299995422363,6147900,FIS,United States
1118596,84879EDE-767B-DC5E-A377-7C3F92603B1C,2026-03-18 00:00:00,49.2099990844727,5492200,FIS,United States
1118597,84879EDE-767B-DC5E-A377-7C3F92603B1C,2026-03-19 00:00:00,49.2599983215332,5519600,FIS,United States


In [3]:
symbols = prices["Symbol"].unique()

# Train a model to predict the next day's price based on the previous 30 days of prices, for each stock symbol separately.
window_size = 30
models_by_symbol = {}
losses_by_symbol = {}
n_models_to_train = 3
for symbol in tqdm(symbols[:3]):
    if symbol in models_by_symbol and len(models_by_symbol[symbol]) >= n_models_to_train: continue # Resume training
    symbol_prices = prices[prices["Symbol"] == symbol].sort_values("Timestamp")
    _prices = np.array(symbol_prices["Price"].values).astype(np.float32)
    logreturns = np.diff(np.log(_prices))
    if len(symbol_prices) < window_size + 1:
        continue

    X = []
    y = []
    for i in range(len(symbol_prices) - window_size - 1):
        X.append(logreturns[i:i+window_size])
        y.append(logreturns[i+window_size] > 0)  # Binary classification: will the price go up or down?

    X = np.array(X)
    y = utils.to_categorical(y, num_classes=2)

    models_by_symbol[symbol] = []
    losses_by_symbol[symbol] = []
    for _ in range(n_models_to_train):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)
        # Shuffle the training data
        X_train, y_train = shuffle(X_train, y_train)

        model = models.Sequential([
            layers.Input(shape=(window_size,)),
            layers.Dense(32),
            layers.LeakyReLU(),
            layers.Dense(64),
            layers.LeakyReLU(),
            layers.Dense(2, activation='softmax')
        ])
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(X_train, y_train, epochs=25, batch_size=32, verbose=0, validation_data=(X_test, y_test))
        models_by_symbol[symbol].append(model)


        # Calculate the loss
        loss = model.evaluate(X_test, y_test, verbose=0)
        losses_by_symbol[symbol].append(loss)


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:41<00:00, 13.80s/it]


In [ ]:
out_dir = '../StocksPlatform/Services/Analysis/Models/Price'
os.makedirs(out_dir, exist_ok=True)
# models_by_symbol["TSLA"][1].export(os.path.join(out_dir, "tsla_model_1.onnx"))
for symbol in models_by_symbol:
    for i, model in enumerate(models_by_symbol[symbol]):
        # onnx_model, _ = tf2onnx.convert.from_keras(model)
        # onnx.save(onnx_model, os.path.join(out_dir, f"{symbol.lower()}_model_{i}.onnx"))
        models_by_symbol[symbol][i].export(os.path.join(out_dir, f"{symbol.lower()}_model_{i}.onnx"))

In [50]:
# Save all models as h5
for symbol, models in models_by_symbol.items():
    for i, model in enumerate(models):
        model.save(f'Models/{symbol}_model_{i}.h5')

c:\Users\Jon\anaconda3\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [51]:
# Save all losses as csv
losses_df = pd.DataFrame([
    {"Symbol": symbol, "ModelIndex": i, "Loss": loss[0], "Accuracy": loss[1]}
    for symbol, losses in losses_by_symbol.items()
    for i, loss in enumerate(losses)
])
losses_df.to_csv('model_losses.csv', index=False)

--- Restarting kernel ---

In [5]:
# Read models back in
models_by_symbol = {}
symbols = prices["Symbol"].unique()
for symbol in tqdm(symbols):
    models_by_symbol[symbol] = []
    for i in range(3):
        try:
            model = models.load_model(f'Models/{symbol}_model_{i}.h5')
            models_by_symbol[symbol].append(model)
        except Exception as e:
            print(f"Could not load model for {symbol} index {i}: {e}")


  4%|▍         | 22/535 [00:08<03:27,  2.47it/s]

Could not load model for GKP index 0: No file or directory found at Models/GKP_model_0.h5
Could not load model for GKP index 1: No file or directory found at Models/GKP_model_1.h5
Could not load model for GKP index 2: No file or directory found at Models/GKP_model_2.h5


  6%|▌         | 32/535 [00:12<03:24,  2.46it/s]

Could not load model for PLGC index 0: No file or directory found at Models/PLGC_model_0.h5
Could not load model for PLGC index 1: No file or directory found at Models/PLGC_model_1.h5
Could not load model for PLGC index 2: No file or directory found at Models/PLGC_model_2.h5


100%|██████████| 535/535 [01:49<00:00,  4.87it/s] 

Could not load model for IDXX index 0: No file or directory found at Models/IDXX_model_0.h5
Could not load model for IDXX index 1: No file or directory found at Models/IDXX_model_1.h5
Could not load model for IDXX index 2: No file or directory found at Models/IDXX_model_2.h5
Could not load model for FITB index 0: No file or directory found at Models/FITB_model_0.h5
Could not load model for FITB index 1: No file or directory found at Models/FITB_model_1.h5
Could not load model for FITB index 2: No file or directory found at Models/FITB_model_2.h5
Could not load model for IP index 0: No file or directory found at Models/IP_model_0.h5
Could not load model for IP index 1: No file or directory found at Models/IP_model_1.h5
Could not load model for IP index 2: No file or directory found at Models/IP_model_2.h5
Could not load model for HSIC index 0: No file or directory found at Models/HSIC_model_0.h5
Could not load model for HSIC index 1: No file or directory found at Models/HSIC_model_1.h5


In [ ]:
import os
import shutil
import tempfile
import tf2onnx
import onnx

out_dir = '../StocksPlatform/Services/Analysis/Models/Price'
os.makedirs(out_dir, exist_ok=True)

exported = 0
for symbol, sym_models in models_by_symbol.items():
    for i, model in enumerate(sym_models):
        # Save as SavedModel to a temp dir, then convert — avoids the Windows
        # access-violation crash that occurs in from_keras's tracing path.
        with tempfile.TemporaryDirectory() as tmp:
            model.export(tmp)
            onnx_model, _ = tf2onnx.convert.from_saved_model(
                tmp,
                input_names=['input'],
                output_names=['output'],
                opset=13,
            )
        path = os.path.join(out_dir, f'model_{symbol}_{i}.onnx')
        onnx.save_model(onnx_model, path)
        exported += 1

print(f"Exported {exported} model(s) to {out_dir}")


INFO:tensorflow:Assets written to: C:\Users\Jon\AppData\Local\Temp\tmp0ltg475x\assets


INFO:tensorflow:Assets written to: C:\Users\Jon\AppData\Local\Temp\tmp0ltg475x\assets


Saved artifact at 'C:\Users\Jon\AppData\Local\Temp\tmp0ltg475x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30), dtype=tf.float32, name='input_38')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1735163886960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1733039507760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1733039514800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1733039510928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1733039517792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1733039513744: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [7]:
import json

window_size = 30

# Build per-symbol log-return series (all available history, not just the training window)
symbol_returns = {}
for symbol in symbols:
    sp = prices[prices["Symbol"] == symbol].sort_values("Timestamp")
    p = np.array(sp["Price"].values, dtype=np.float64)
    if len(p) < 2:
        continue
    lr = np.diff(np.log(p))
    if len(lr) >= window_size:
        symbol_returns[symbol] = lr

# Compute pairwise Pearson correlation; align series by taking the last min(len_a, len_b) values
sym_list = list(symbol_returns.keys())
corr_matrix = {}
for sym_a in sym_list:
    corr_matrix[sym_a] = {}
    for sym_b in sym_list:
        if sym_a == sym_b:
            corr_matrix[sym_a][sym_b] = 1.0
            continue
        a, b = symbol_returns[sym_a], symbol_returns[sym_b]
        n = min(len(a), len(b))
        if n < window_size:
            corr_matrix[sym_a][sym_b] = 0.0
        else:
            corr = float(np.corrcoef(a[-n:], b[-n:])[0, 1])
            corr_matrix[sym_a][sym_b] = round(corr if not np.isnan(corr) else 0.0, 4)

models_dir = '../StocksPlatform/Services/Analysis/Models'
os.makedirs(models_dir, exist_ok=True)
corr_path = os.path.join(models_dir, 'correlation.json')
with open(corr_path, 'w') as f:
    json.dump(corr_matrix, f)

print(f"Saved correlation matrix ({len(sym_list)} symbols) to {corr_path}")

Saved correlation matrix (533 symbols) to ../StocksPlatform/Services/Analysis/Models\correlation.json
